# Herald Analysis — Worlds 2026

Investigating the relationship between Rift Herald acquisition, acquisition timing,
team side (Blue/Red), and match outcomes.

**Dataset:** `worlds_swiss_onward.csv`  
**Source:** Oracle's Elixir  
**Scope:** Worlds 2026, Swiss Stage onward

In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Find the project root by looking for the data folder
current_path = Path.cwd()

PROJECT_ROOT = next(
    (path for path in [current_path, *current_path.parents]
     if (path / "data" / "processed").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project folder. "
        "Make sure this notebook is inside the project."
    )

# Define the processed dataset path
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "worlds_swiss_onward.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\boyle\League_Of_Legends_Herald_Analysis
Dataset path: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed\worlds_swiss_onward.csv
Dataset exists: True


In [3]:
# Load the prepared Worlds dataset
df = pd.read_csv(DATA_PATH)

# Convert date column back to datetime
df["date"] = pd.to_datetime(df["date"])

print("Rows:", len(df))
print("Unique games:", df["gameid"].nunique())

display(df.head())

Rows: 864
Unique games: 72


,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,LOLTMNT02_482269,complete,NaN,WLDs,2026,NaN,0,2026-10-23 17:21:18,1,16.2,...,209.0,-1155.0,-2725.0,-20.0,1.0,5.0,4.0,1.0,8.0,2.0
1,LOLTMNT02_482269,complete,NaN,WLDs,2026,NaN,0,2026-10-23 17:21:18,1,16.2,...,244.0,-4661.0,-5372.0,-70.0,0.0,7.0,2.0,4.0,10.0,1.0
2,LOLTMNT02_482269,complete,NaN,WLDs,2026,NaN,0,2026-10-23 17:21:18,1,16.2,...,255.0,-3108.0,-3966.0,-39.0,4.0,4.0,4.0,7.0,6.0,1.0
3,LOLTMNT02_482269,complete,NaN,WLDs,2026,NaN,0,2026-10-23 17:21:18,1,16.2,...,235.0,-370.0,-1539.0,18.0,4.0,4.0,1.0,4.0,8.0,2.0
4,LOLTMNT02_482269,complete,NaN,WLDs,2026,NaN,0,2026-10-23 17:21:18,1,16.2,...,29.0,-501.0,-2401.0,0.0,0.0,8.0,5.0,0.0,13.0,3.0


In [4]:
# Find all columns containing "herald"
herald_columns = [
    column for column in df.columns
    if "herald" in column.lower()
]

print("Herald-related columns:")
print(herald_columns)

Herald-related columns:
['firstherald', 'heralds', 'opp_heralds']


In [5]:
# Keep only team-level records
team_df = df[df["position"] == "team"].copy()

# Inspect the relevant columns
columns_to_inspect = [
    "gameid",
    "date",
    "game",
    "teamname",
    "side",
    "result",
    "gamelength",
    "firstherald",
    "heralds",
    "opp_heralds"
]

display(team_df[columns_to_inspect].head(20))

,gameid,date,game,teamname,side,result,gamelength,firstherald,heralds,opp_heralds
10,LOLTMNT02_482269,2026-10-23 17:21:18,1,Invictus Gaming,Blue,0,1561,1.0,1.0,0.0
11,LOLTMNT02_482269,2026-10-23 17:21:18,1,Gen.G,Red,1,1561,0.0,0.0,1.0
22,LOLTMNT02_483314,2026-10-23 18:23:03,1,Top Esports,Blue,0,1431,0.0,0.0,1.0
23,LOLTMNT02_483314,2026-10-23 18:23:03,1,Hanwha Life Esports,Red,1,1431,1.0,1.0,0.0
34,LOLTMNT02_483342,2026-10-23 19:12:48,1,Movistar KOI,Blue,1,2321,0.0,0.0,1.0
35,LOLTMNT02_483342,2026-10-23 19:12:48,1,Bilibili Gaming,Red,0,2321,1.0,1.0,0.0
46,LOLTMNT02_482292,2026-10-23 20:16:06,1,Karmine Corp,Blue,1,1996,0.0,0.0,1.0
47,LOLTMNT02_482292,2026-10-23 20:16:06,1,Anyone's Legend,Red,0,1996,1.0,1.0,0.0
58,LOLTMNT02_482304,2026-10-23 21:16:28,1,Dplus Kia,Blue,1,1958,0.0,0.0,1.0
59,LOLTMNT02_482304,2026-10-23 21:16:28,1,G2 Esports,Red,0,1958,1.0,1.0,0.0


In [6]:
# Inspect the match URLs already available
urls = (
    df[["gameid", "url"]]
    .drop_duplicates()
    .dropna()
)

print("Unique game URLs:", len(urls))
display(urls.head(10))

Unique game URLs: 0


,gameid,url


In [7]:
%pip install selenium beautifulsoup4 pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ------------------------- -------------- 7.6/12.1 MB 47.5 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 52.8 MB/s  0:00:00
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)

   ---------- ----------------------------- 2/8 [sniffio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- -------------- 5/8 [trio]
   ------------------------- ------------


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import re
import time
import pandas as pd

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
# Configure Chrome
options = webdriver.ChromeOptions()

# Launch browser
driver = webdriver.Chrome(options=options)

print("Browser successfully launched!")

Browser successfully launched!


In [12]:
# First Worlds match
GAME_ID = 83638

url = f"https://gol.gg/game/stats/{GAME_ID}/page-game/"

driver.get(url)

# Wait for the page to load
WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: Gen.G vs IG game 1 - Worlds 2026 Main Event ROUND1 - Games of Legends
Current URL: https://gol.gg/game/stats/83638/page-game/


In [13]:
# Parse the rendered page
soup = BeautifulSoup(driver.page_source, "html.parser")

# Find objective timeline events
objective_elements = soup.select(
    "span.red_action, span.blue_action"
)

print("Objective elements found:", len(objective_elements))

# Display the raw HTML for the first 20
for element in objective_elements[:20]:
    print(element)
    print("-" * 60)

Objective elements found: 8
<span class="red_action">6:14<br/><img alt="Infernal Drake" class="champion_icon_light" src="../_img/fire-dragon.png"/></span>
------------------------------------------------------------
<span class="blue_action"><img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/><br/>6:15</span>
------------------------------------------------------------
<span class="red_action">12:10<br/><img alt="Ocean Drake" class="champion_icon_light" src="../_img/ocean-dragon.png"/></span>
------------------------------------------------------------
<span class="blue_action"><img alt="First Tower" class="champion_icon_light" src="../_img/firsttower2.png"/><br/>14:55</span>
------------------------------------------------------------
<span class="blue_action"><img alt="Rift Herald" class="champion_icon_light" src="../_img/herald-icon.png"/><br/>15:20</span>
------------------------------------------------------------
<span class="red_action">17:32<br/><i

In [14]:
# Inspect page title and headings
print("PAGE TITLE:")
print(soup.title.get_text(strip=True))

print("\nHEADINGS:")
for heading in soup.find_all(["h1", "h2", "h3", "h4"]):
    print(heading.get_text(" ", strip=True))

print("\nTEAM NAME REFERENCES:")
for team in ["Gen.G", "Invictus Gaming"]:
    matches = soup.find_all(
        string=lambda text: text and team.lower() in text.lower()
    )

    print(f"\n{team}:")
    for match in matches[:5]:
        print(match.parent.prettify()[:1000])

PAGE TITLE:
Gen.G vs IG game 1 - Worlds 2026 Main Event ROUND1 - Games of Legends

HEADINGS:
Support Games of Legends on Patreon and enjoy an ad-free experience:
Gen.G vs IG
26:01

TEAM NAME REFERENCES:

Gen.G:
<title>
 Gen.G vs IG game 1 - Worlds 2026 Main Event ROUND1 - Games of Legends
</title>

<script>
 var search_change = function(){
        return function(){
        
        var itemid = $('#selectSearch').val().split('_');        
        
        if( itemid[0] == 'p'){                   
          window.location.replace('../players/player-stats/'+itemid[1]+'/season-S16/split-Summer/tournament-ALL/champion-ALL/') ;
        }
        if( itemid[0] == 'c'){                   
          window.location.replace('../champion/champion-stats/'+itemid[1]+'/season-S16/split-Summer/tournament-ALL/') ;
        }
        if( itemid[0] == 't'){                   
          window.location.replace('../teams/team-stats/'+itemid[1]+'/split-Summer/tournament-ALL/') ;
        }
       }       

In [16]:
# Extract Blue and Red team names and results

team_mapping = {}

for side, css_class in [
    ("Blue", "blue-line-header"),
    ("Red", "red-line-header")
]:

    element = soup.select_one(f".{css_class}")

    if element:
        team_name = element.find("a").get_text(strip=True)
        result_text = element.get_text(" ", strip=True)

        result = (
            "WIN" if "WIN" in result_text
            else "LOSS" if "LOSS" in result_text
            else None
        )

        team_mapping[side] = {
            "team": team_name,
            "result": result
        }

print(team_mapping)

{'Blue': {'team': 'Invictus Gaming', 'result': 'LOSS'}, 'Red': {'team': 'Gen.G', 'result': 'WIN'}}


In [17]:
# Extract all timeline events from the match

events = []

# Find all timeline events belonging to either team
timeline_elements = soup.select(
    "span.red_action, span.blue_action"
)

for element in timeline_elements:

    # Determine team side
    if "blue_action" in element.get("class", []):
        side = "Blue"
    else:
        side = "Red"

    # Look up team information
    team_info = team_mapping.get(side, {})

    # Extract event timestamp
    text = element.get_text(" ", strip=True)

    time_match = re.search(
        r"\b\d{1,2}:\d{2}\b",
        text
    )

    event_time = time_match.group() if time_match else None

    # Extract event name
    image = element.find("img")

    event_name = (
        image.get("alt")
        if image
        else None
    )

    # Convert timestamp into seconds for easier analysis
    if event_time:
        minutes, seconds = map(int, event_time.split(":"))
        time_seconds = minutes * 60 + seconds
    else:
        time_seconds = None

    events.append({
        "game_id": GAME_ID,
        "time": event_time,
        "time_seconds": time_seconds,
        "event": event_name,
        "side": side,
        "team": team_info.get("team"),
        "result": team_info.get("result")
    })

# Create DataFrame
events_df = pd.DataFrame(events)

display(events_df)

,game_id,time,time_seconds,event,side,team,result
0,83638,6:14,374,Infernal Drake,Red,Gen.G,WIN
1,83638,6:15,375,First blood,Blue,Invictus Gaming,LOSS
2,83638,12:10,730,Ocean Drake,Red,Gen.G,WIN
3,83638,14:55,895,First Tower,Blue,Invictus Gaming,LOSS
4,83638,15:20,920,Rift Herald,Blue,Invictus Gaming,LOSS
5,83638,17:32,1052,Cloud Drake,Red,Gen.G,WIN
6,83638,22:50,1370,Cloud Drake,Red,Gen.G,WIN
7,83638,23:38,1418,Nashor,Red,Gen.G,WIN


In [18]:
# Define output path
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / f"golgg_events_{GAME_ID}.csv"
)

# Save CSV
events_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Total events:", len(events_df))

Saved: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed\golgg_events_83638.csv
Total events: 8


In [19]:

def scrape_golgg_game(game_id, driver):
    """
    Scrape timeline events and team information for one Gol.gg game.
    Returns a DataFrame containing all extracted events.
    """

    url = f"https://gol.gg/game/stats/{game_id}/page-game/"

    # Open the match page
    driver.get(url)

    # Wait for the page body to load
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Identify the Blue and Red teams
    team_mapping = {}

    for side, css_class in [
        ("Blue", "blue-line-header"),
        ("Red", "red-line-header")
    ]:
        element = soup.select_one(f".{css_class}")

        if element and element.find("a"):
            team_name = element.find("a").get_text(strip=True)
            result_text = element.get_text(" ", strip=True)

            result = (
                "WIN" if "WIN" in result_text
                else "LOSS" if "LOSS" in result_text
                else None
            )

            team_mapping[side] = {
                "team": team_name,
                "result": result
            }

    # Find all timeline events
    timeline_elements = soup.select(
        "span.red_action, span.blue_action"
    )

    if not timeline_elements:
        raise ValueError(
            f"No timeline events found for game {game_id}"
        )

    # Extract individual events
    events = []

    for element in timeline_elements:

        # Determine which side performed the event
        if "blue_action" in element.get("class", []):
            side = "Blue"
        else:
            side = "Red"

        team_info = team_mapping.get(side, {})

        # Extract event time
        text = element.get_text(" ", strip=True)

        time_match = re.search(
            r"\b\d{1,2}:\d{2}\b",
            text
        )

        event_time = (
            time_match.group()
            if time_match
            else None
        )

        # Extract event name from image alt text
        image = element.find("img")

        event_name = (
            image.get("alt")
            if image
            else None
        )

        # Convert time to total seconds
        if event_time:
            minutes, seconds = map(
                int,
                event_time.split(":")
            )
            time_seconds = minutes * 60 + seconds
        else:
            time_seconds = None

        # Store the event
        events.append({
            "game_id": game_id,
            "time": event_time,
            "time_seconds": time_seconds,
            "event": event_name,
            "side": side,
            "team": team_info.get("team"),
            "result": team_info.get("result")
        })

    return pd.DataFrame(events)


In [20]:

# Game IDs for our first test batch
game_ids = range(83638, 83643)

# Store the results from each game
all_events = []
failed_games = []

for game_id in game_ids:

    print(f"Scraping game {game_id}...")

    try:
        game_events = scrape_golgg_game(
            game_id,
            driver
        )

        all_events.append(game_events)

        print(
            f"  Found {len(game_events)} events"
        )

    except Exception as error:
        print(f"  Failed: {error}")
        failed_games.append(game_id)

        # Stop if the page indicates access is restricted
        page_text = driver.page_source.lower()

        if any(term in page_text for term in [
            "403 forbidden",
            "access denied",
            "request blocked"
        ]):
            print("Access restriction detected. Stopping scraper.")
            break

    # Pause before the next page
    time.sleep(3)


# Combine all successful games
if all_events:
    combined_events_df = pd.concat(
        all_events,
        ignore_index=True
    )
else:
    combined_events_df = pd.DataFrame()


print("\nScraping complete.")
print("Successful games:", combined_events_df["game_id"].nunique()
      if not combined_events_df.empty else 0)
print("Failed games:", failed_games)
print("Total events:", len(combined_events_df))


Scraping game 83638...
  Found 8 events
Scraping game 83639...
  Found 10 events
Scraping game 83640...
  Found 10 events
Scraping game 83641...
  Found 9 events
Scraping game 83642...
  Found 7 events

Scraping complete.
Successful games: 5
Failed games: []
Total events: 44


In [21]:
display(
    combined_events_df.sort_values(
        ["game_id", "time_seconds"]
    ).reset_index(drop=True)
)

,game_id,time,time_seconds,event,side,team,result
0,83638,6:14,374,Infernal Drake,Red,Gen.G,WIN
1,83638,6:15,375,First blood,Blue,Invictus Gaming,LOSS
2,83638,12:10,730,Ocean Drake,Red,Gen.G,WIN
3,83638,14:55,895,First Tower,Blue,Invictus Gaming,LOSS
4,83638,15:20,920,Rift Herald,Blue,Invictus Gaming,LOSS
5,83638,17:32,1052,Cloud Drake,Red,Gen.G,WIN
6,83638,22:50,1370,Cloud Drake,Red,Gen.G,WIN
7,83638,23:38,1418,Nashor,Red,Gen.G,WIN
8,83639,6:34,394,Hextech Drake,Red,CTBC Flying Oyster,WIN
9,83639,9:12,552,First blood,Red,CTBC Flying Oyster,WIN


In [23]:
event_counts = (
    combined_events_df
    .groupby("game_id")
    .size()
    .reset_index(name="event_count")
)

display(event_counts)

display(
    combined_events_df["event"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,game_id,event_count
0,83638,8
1,83639,10
2,83640,10
3,83641,9
4,83642,7


,count
event,
Ocean Drake,7
Nashor,6
First blood,5
First Tower,5
Rift Herald,5
Mountain Drake,5
Infernal Drake,4
Hextech Drake,3
Cloud Drake,2


In [24]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "golgg_events_first_5_games.csv"
)

combined_events_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed\golgg_events_first_5_games.csv


In [25]:

# Remaining candidate game IDs
game_ids = range(83643, 83722)

# Store successful results and failures
remaining_events = []
failed_games = []

# Save progress every 10 successful games
for game_id in game_ids:

    print(f"Scraping game {game_id}...")

    try:
        game_events = scrape_golgg_game(
            game_id,
            driver
        )

        remaining_events.append(game_events)

        print(f"  Found {len(game_events)} events")

    except Exception as error:
        print(f"  Failed: {error}")
        failed_games.append(game_id)

        page_text = driver.page_source.lower()

        if any(term in page_text for term in [
            "403 forbidden",
            "access denied",
            "request blocked",
            "429 too many requests"
        ]):
            print("Access restriction detected. Stopping.")
            break

    # Combine successful results so far
    if remaining_events:
        progress_df = pd.concat(
            remaining_events,
            ignore_index=True
        )

        # Save progress periodically
        if len(remaining_events) % 10 == 0:
            progress_path = (
                PROJECT_ROOT
                / "data"
                / "processed"
                / "golgg_events_remaining_progress.csv"
            )

            progress_df.to_csv(
                progress_path,
                index=False
            )

            print("  Progress saved.")

    # Pause between requests
    time.sleep(3)


# Combine all remaining successful games
if remaining_events:
    remaining_events_df = pd.concat(
        remaining_events,
        ignore_index=True
    )
else:
    remaining_events_df = pd.DataFrame()

print("\nBatch complete.")
print("Successful games:", remaining_events_df["game_id"].nunique()
      if not remaining_events_df.empty else 0)
print("Failed games:", failed_games)
print("Total events:", len(remaining_events_df))


Scraping game 83643...
  Found 7 events
Scraping game 83644...
  Found 7 events
Scraping game 83645...
  Found 10 events
Scraping game 83646...
  Found 9 events
Scraping game 83647...
  Found 9 events
Scraping game 83648...
  Found 10 events
Scraping game 83649...
  Found 10 events
Scraping game 83650...
  Found 8 events
Scraping game 83651...
  Found 8 events
Scraping game 83652...
  Found 11 events
  Progress saved.
Scraping game 83653...
  Found 8 events
Scraping game 83654...
  Found 8 events
Scraping game 83655...
  Found 10 events
Scraping game 83656...
  Found 10 events
Scraping game 83657...
  Found 7 events
Scraping game 83658...
  Failed: No timeline events found for game 83658
Scraping game 83659...
  Found 10 events
Scraping game 83660...
  Found 8 events
Scraping game 83661...
  Found 8 events
Scraping game 83662...
  Found 7 events
Scraping game 83663...
  Found 10 events
  Progress saved.
Scraping game 83664...
  Found 11 events
Scraping game 83665...
  Found 11 events
S

In [26]:

# Combine the first five games with the new batch
all_games_df = pd.concat(
    [combined_events_df, remaining_events_df],
    ignore_index=True
)

# Sort by game and event time
all_games_df = all_games_df.sort_values(
    ["game_id", "time_seconds"]
).reset_index(drop=True)

# Save the consolidated dataset
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "worlds_objective_events.csv"
)

all_games_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved dataset to: {OUTPUT_PATH}")
print("Unique games:", all_games_df["game_id"].nunique())
print("Total events:", len(all_games_df))


Saved dataset to: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\processed\worlds_objective_events.csv
Unique games: 72
Total events: 670


In [27]:
print(sorted(all_games_df["game_id"].unique()))

[np.int64(83638), np.int64(83639), np.int64(83640), np.int64(83641), np.int64(83642), np.int64(83643), np.int64(83644), np.int64(83645), np.int64(83646), np.int64(83647), np.int64(83648), np.int64(83649), np.int64(83650), np.int64(83651), np.int64(83652), np.int64(83653), np.int64(83654), np.int64(83655), np.int64(83656), np.int64(83657), np.int64(83659), np.int64(83660), np.int64(83661), np.int64(83662), np.int64(83663), np.int64(83664), np.int64(83665), np.int64(83667), np.int64(83668), np.int64(83669), np.int64(83670), np.int64(83672), np.int64(83673), np.int64(83674), np.int64(83675), np.int64(83676), np.int64(83678), np.int64(83679), np.int64(83680), np.int64(83681), np.int64(83682), np.int64(83683), np.int64(83684), np.int64(83685), np.int64(83686), np.int64(83687), np.int64(83689), np.int64(83690), np.int64(83691), np.int64(83692), np.int64(83693), np.int64(83694), np.int64(83695), np.int64(83696), np.int64(83697), np.int64(83699), np.int64(83700), np.int64(83701), np.int64(8370

In [28]:
display(
    all_games_df.groupby("game_id")
    .size()
    .reset_index(name="event_count")
)

,game_id,event_count
0,83638,8
1,83639,10
2,83640,10
3,83641,9
4,83642,7
...,...,...
67,83715,10
68,83718,8
69,83719,10
70,83720,11


In [29]:
display(
    all_games_df["event"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
event,
Nashor,100
Infernal Drake,81
First blood,72
First Tower,72
Rift Herald,71
Mountain Drake,68
Hextech Drake,57
Ocean Drake,46
Chemtech Drake,45


In [30]:
display(
    all_games_df.isna().sum().to_frame("missing_values")
)

,missing_values
game_id,0
time,0
time_seconds,0
event,0
side,0
team,0
result,0
